# MAMA-MIA — 02: Preprocessing (v2, izotrop)

20.28 GB xom ma'lumotni bir marta qayta ishlab, Kaggle Dataset sifatida
saqlanadigan ixcham versiyaga aylantiramiz. Keyingi barcha notebook'lar
shu natijadan foydalanadi — 20 GB qayta yuklanmaydi.

## v1'dagi xato va uning tuzatilishi

Birinchi versiya kesim o'qini 2-o'q deb qattiq yozgan edi. Dry run buni ochdi:

| Manba | Shakl | Spacing | Haqiqiy kesim o'qi |
|---|---|---|---|
| DUKE | 448x448x160 | 0.804, 0.804, **1.100** | 2-o'q (aksial) |
| ISPY2 | 256x256x80 | 0.664, 0.664, **2.000** | 2-o'q (aksial) |
| ISPY1 | 60x256x256 | **2.000**, 0.781, 0.781 | **0-o'q** (sagittal) |
| NACT | 60x256x256 | **2.000**, 0.781, 0.781 | **0-o'q** (sagittal) |

Ya'ni ISPY1/NACT uchun kod boshqa anatomik tekislikni kesib, 60 pikselni
256 ga ~4 baravar cho'zgan bo'lar edi. Model DUKE'dan aksial, ISPY1'dan
cho'zilgan koronal rasm ko'rardi — tekislik manba bilan chalkashib,
markazlararo xulosalar ma'nosiz bo'lib qolardi.

**Ikkinchi topilma (v2 dry run'dan):** DUKE ikki ko'krakli, I-SPY va NACT esa
bitta ko'krakli (L-R yo'nalishda atigi ~120 mm). Shu sababli I-SPY tasvirlari
kadrning kichik qismini egallab, atrofi qora to'ldirish bo'lib qolardi — model
buni manba belgisi sifatida osongina o'rganib oladi. Yechim: ikki ko'krakli
hajmlar o'rtadan bo'linadi, har bir holat bitta ko'krakka aylanadi.

**Yechim: izotrop resampling + yagona anatomik tekislik.**

| Qadam | Sabab |
|---|---|
| `as_closest_canonical` → RAS | O'q tartibi barcha holatlar uchun bir xil bo'ladi |
| 1.5 mm izotrop resampling | Spacing heterogenligi yo'qoladi; 2-o'q har doim aksial |
| Tekislikda **markaziy kesish/to'ldirish** (resize emas) | Fizik masshtab (mm/piksel) barcha manbalarda aynan bir xil qoladi |
| **Bitta ko'krakka bo'lish** (DUKE ikki ko'krakli) | Aks holda model qora to'ldirish shaklidan manbani taniydi — FOV farqi soxta signal bo'lib qoladi |
| Kesish oynasi **to'qima markaziga** qo'yiladi | Kadr markaziga emas: manbalar orasida joylashuv farqi kamayadi |
| O'sma bor kesimlar ± margin | Hajmni kamaytiradi, margin manfiy namunalar beradi |
| Intensivlik xomligicha | Harmonizatsiya — tadqiqotning asosiy hissasi, hozir qotirilmaydi |
| Manba bo'yicha HDF5 | `/kaggle/working` da fayllar soni cheklangan |

**Kaggle sozlamasi:** Accelerator = None (CPU), Internet = On.

## 1. Sozlamalar

In [ ]:
!pip install -q huggingface_hub nibabel h5py

import os, re, gc, time, json, shutil, zipfile, tempfile, collections
import numpy as np
import nibabel as nib
import h5py
from scipy.ndimage import zoom as ndzoom
from skimage.filters import threshold_otsu
from huggingface_hub import hf_hub_download

REPO_ID = "YongchengYAO/MAMA-MIA-Lite"
PARTS   = ["data-part001.zip", "data-part002.zip"]

TARGET_SPACING = 1.5    # mm, izotrop
OUT_SIZE       = 192    # tekislikdagi piksel  -> 192 * 1.5 = 288 mm FOV (bitta ko'krak uchun yetarli)
MARGIN         = 8      # o'sma diapazonidan tashqariga qo'shiladigan kesimlar
BILATERAL_MM   = 256    # L-R to'qima kengligi shundan katta bo'lsa -> ikki ko'krakli.
                        # Qiymat HAQIQIY taqsimotdan olingan (1506 holat, v3 ishlovi):
                        # eng katta bo'shliq 247.5 -> 265.5 mm (18 mm), undan pastda
                        # 1037 holat, yuqorida 469. DUKE ning eng kichigi 265.5 mm.
                        # Oldingi 180 mm qiymati 129 holatni (8.6%) xato bo'lib yuborgan edi.

# DRY_RUN: None = hammasi; son = sinov rejimi.
# Commit uchun None qilib qo'yilgan — notebook boshidan oxirigacha o'tadi.
DRY_RUN = None

TMP = "/kaggle/temp" if os.path.isdir("/kaggle/temp") else tempfile.gettempdir()
ZIPDIR = os.path.join(TMP, "zips")
OUTDIR = "/kaggle/working/mamamia_proc" if os.path.isdir("/kaggle/working") else "./mamamia_proc"
scratch = os.path.join(TMP, "scratch")
for d in (ZIPDIR, OUTDIR, scratch):
    os.makedirs(d, exist_ok=True)

print("Zip papkasi :", ZIPDIR)
print("Chiqish     :", OUTDIR)
print(f"Izotrop     : {TARGET_SPACING} mm,  FOV = {OUT_SIZE*TARGET_SPACING:.0f} mm")
print(f"Bo'lish chegarasi: L-R to'qima > {BILATERAL_MM} mm -> ikki ko'krakli")
print("Rejim       :", "DRY RUN" if DRY_RUN else "TO'LIQ")

## 2. Arxivlarni yuklash

Ikkala zip `/kaggle/temp` ga tushadi — 20 GB working kvotasiga kirmaydi.
Seans tugagach yo'qoladi, shuning uchun natijani Dataset qilib saqlaymiz.

In [ ]:
t0 = time.time()
zip_paths = []
for p in PARTS:
    print("yuklanmoqda:", p, "...", flush=True)
    path = hf_hub_download(REPO_ID, p, repo_type="dataset", local_dir=ZIPDIR)
    zip_paths.append(path)
    print(f"   ok  {os.path.getsize(path)/2**30:.2f} GB")
print(f"\nJami {time.time()-t0:.0f} s")

!df -h {TMP} | tail -1

## 3. Indeks qurish

In [ ]:
zips = {p: zipfile.ZipFile(p) for p in zip_paths}

index = {}
for p, z in zips.items():
    for n in z.namelist():
        if not n.endswith(".nii.gz"):
            continue
        cid = os.path.basename(n)[:-len(".nii.gz")]
        slot = "img" if n.startswith("Images/") else "msk" if n.startswith("Masks/") else None
        if slot is None:
            continue
        index.setdefault(cid, {})[slot] = (p, n)

complete = {c: v for c, v in index.items() if "img" in v and "msk" in v}
print(f"To'liq juftliklar: {len(complete)} / {len(index)}")

def source_of(cid):
    """DUKE_001 -> DUKE,  ISPY1_1001 -> ISPY1  (ISPY1 va ISPY2 ajratiladi:
    ular turli sinovlar, turli davr va protokol)."""
    return cid.split("_")[0].upper()

print("\nManba taqsimoti:")
for s, n in sorted(collections.Counter(source_of(c) for c in complete).items(),
                   key=lambda x: -x[1]):
    print(f"  {s:<8} {n:>5}")

## 4. Ishlov funksiyalari

Tartib: RAS kanonik yo'nalish → 1.5 mm izotrop → o'sma bo'yicha z-kesish →
tekislikda markaziy kesish/to'ldirish → `uint16`.

In [ ]:
def load_nii_from_zip(zpath, member, scratch):
    """Zip ichidagi .nii.gz ni vaqtincha faylga yozib, RAS kanonik holda o'qiydi."""
    dst = os.path.join(scratch, "tmp_" + os.path.basename(member))
    with zips[zpath].open(member) as src, open(dst, "wb") as out:
        shutil.copyfileobj(src, out, length=1 << 20)
    obj = nib.load(dst)
    raw_shape = tuple(int(x) for x in obj.shape[:3])
    raw_zooms = tuple(float(x) for x in obj.header.get_zooms()[:3])
    raw_ax    = "".join(nib.aff2axcodes(obj.affine))

    can = nib.as_closest_canonical(obj)          # -> RAS: 0=L-R, 1=A-P, 2=S-I
    arr = np.asanyarray(can.dataobj)
    zooms = tuple(float(x) for x in can.header.get_zooms()[:3])
    del obj, can
    os.remove(dst)
    return arr, zooms, raw_shape, raw_zooms, raw_ax


def resample_iso(img, msk, zooms, target=TARGET_SPACING):
    """1.5 mm izotrop. Rasm uchun chiziqli, maska uchun eng yaqin qo'shni."""
    factors = tuple(z / target for z in zooms)
    if all(abs(f - 1.0) < 1e-3 for f in factors):
        return img.astype(np.float32), msk.astype(np.uint8)
    img_r = ndzoom(img.astype(np.float32), factors, order=1, mode="nearest")
    msk_r = ndzoom(msk.astype(np.uint8),   factors, order=0, mode="nearest")
    return img_r, msk_r


def foreground_bbox(vol):
    """To'qima (havo bo'lmagan) sohasining chegaralarini topadi.
    Otsu chegarasi; u ishlamasa oddiy persentil chegarasiga tushadi."""
    sub = vol[::2, ::2, ::2].astype(np.float32)
    try:
        thr = float(threshold_otsu(sub))
    except Exception:
        thr = float(np.percentile(sub, 60))
    fg = vol > thr
    if not fg.any():
        thr = float(np.percentile(vol, 60))
        fg = vol > thr
        if not fg.any():
            return (0, vol.shape[0], 0, vol.shape[1]), 0.0
    xs = np.where(fg.any(axis=(1, 2)))[0]
    ys = np.where(fg.any(axis=(0, 2)))[0]
    bbox = (int(xs[0]), int(xs[-1]) + 1, int(ys[0]), int(ys[-1]) + 1)
    return bbox, thr


def split_breasts(img, msk, bbox):
    """Ikki ko'krakli hajmni L-R o'qi (0-o'q) bo'yicha o'rtadan bo'ladi.

    Bo'lish nuqtasi — to'qima chegarasining markazi (tana o'rtasi, taxminan
    to'sh suyagi), kadr markazi emas: bemor markazda yotmagan bo'lishi mumkin.
    Faqat o'sma bor yarim(lar) qaytariladi."""
    x0, x1, _, _ = bbox
    mid = (x0 + x1) // 2
    out = []
    for side, sl in (("R", slice(0, mid)), ("L", slice(mid, img.shape[0]))):
        m = msk[sl]
        if m.any():
            out.append((side, img[sl], m))
    return out


def window_fit(vol, size, center_row, pad_value=0.0):
    """Tekislikda `size` o'lchamli oynani kesadi/to'ldiradi — resize QILMAYDI,
    ya'ni fizik masshtab (mm/piksel) o'zgarmaydi.

    0-o'q bo'yicha oyna `center_row` atrofiga, 1-o'q bo'yicha markazga qo'yiladi."""
    h, w = vol.shape[0], vol.shape[1]
    out = np.full((size, size, vol.shape[2]), pad_value, dtype=vol.dtype)

    sy0 = int(round(center_row - size / 2))
    sy0 = max(0, min(sy0, max(0, h - size)))
    sy1 = min(h, sy0 + size)
    sx0 = max(0, (w - size) // 2); sx1 = min(w, sx0 + size)

    dy0 = max(0, (size - (sy1 - sy0)) // 2); dy1 = dy0 + (sy1 - sy0)
    dx0 = max(0, (size - (sx1 - sx0)) // 2); dx1 = dx0 + (sx1 - sx0)
    out[dy0:dy1, dx0:dx1, :] = vol[sy0:sy1, sx0:sx1, :]
    return out


def to_uint16(vol, orig_min, orig_max):
    """ASL hajm diapazoniga qarab uint16 ga joylaydi.

    MUHIM: qaror asl diapazonga qarab qabul qilinadi. Har bir holatni
    min-max bilan cho'zish manbalar orasidagi intensivlik farqini —
    ya'ni o'rganilayotgan domen siljishini — yo'q qilardi."""
    if orig_min >= 0 and orig_max <= 65535:
        return np.clip(np.rint(vol), 0, 65535).astype(np.uint16), 1.0, 0.0, False
    scale = 65535.0 / (orig_max - orig_min) if orig_max > orig_min else 1.0
    out = np.clip(np.rint((vol - orig_min) * scale), 0, 65535).astype(np.uint16)
    return out, scale, float(orig_min), True


def process_case(cid, scratch):
    """Bitta holatdan bir yoki ikkita bitta-ko'krakli namuna qaytaradi.
    Qaytadi: (natijalar ro'yxati, o'tkazish sababi yoki None)"""
    zp_i, n_i = complete[cid]["img"]
    zp_m, n_m = complete[cid]["msk"]
    img, zooms, raw_shape, raw_zooms, raw_ax = load_nii_from_zip(zp_i, n_i, scratch)
    msk, _, _, _, _                          = load_nii_from_zip(zp_m, n_m, scratch)

    base = {"case": cid, "source": source_of(cid)}
    if img.shape != msk.shape:
        return [], {**base, "skip": "shape_mismatch"}
    if not (msk > 0).any():
        return [], {**base, "skip": "empty_mask"}

    i_min, i_max = float(img.min()), float(img.max())
    p1, p50, p99 = (float(np.percentile(img, q)) for q in (1, 50, 99))

    # --- 1.5 mm izotrop: shundan keyin 2-o'q har doim aksial ---
    img_i, msk_i = resample_iso(img, msk, zooms)
    msk_i = (msk_i > 0).astype(np.uint8)
    del img, msk
    if not msk_i.any():
        return [], {**base, "skip": "mask_lost_in_resample"}

    # --- to'qima chegarasi va bitta/ikki ko'kraklik qarori ---
    bbox, thr = foreground_bbox(img_i)
    lr_mm = (bbox[1] - bbox[0]) * TARGET_SPACING
    ap_mm = (bbox[3] - bbox[2]) * TARGET_SPACING
    bilateral = lr_mm > BILATERAL_MM

    parts = (split_breasts(img_i, msk_i, bbox) if bilateral
             else [("W", img_i, msk_i)])
    if not parts:
        return [], {**base, "skip": "no_tumor_side"}

    results = []
    for side, im_p, mk_p in parts:
        zs = np.where(mk_p.any(axis=(0, 1)))[0]
        z0 = max(0, int(zs[0]) - MARGIN)
        z1 = min(im_p.shape[2], int(zs[-1]) + 1 + MARGIN)
        im_c, mk_c = im_p[:, :, z0:z1], mk_p[:, :, z0:z1]

        # oynani shu yarimning to'qima markaziga qo'yamiz
        pb, _ = foreground_bbox(im_c)
        center_row = (pb[0] + pb[1]) / 2.0

        im_f = window_fit(im_c, OUT_SIZE, center_row, pad_value=float(i_min))
        mk_f = window_fit(mk_c, OUT_SIZE, center_row, pad_value=0).astype(np.uint8)
        t_after, t_before = int(mk_f.sum()), int(mk_c.sum())
        if t_after == 0:
            continue

        im_u, scale, offset, scaled = to_uint16(im_f, i_min, i_max)
        key = cid if side == "W" else f"{cid}_{side}"
        fill = float((im_u > thr).mean())

        results.append(((im_u, mk_f), {
            **base, "key": key, "side": side, "bilateral": bool(bilateral),
            "raw_shape": "x".join(map(str, raw_shape)),
            "raw_spacing": ",".join(f"{z:.3f}" for z in raw_zooms),
            "raw_orient": raw_ax,
            "ras_spacing": ",".join(f"{z:.3f}" for z in zooms),
            "slice_axis_raw": int(np.argmax(raw_zooms)),
            "fg_lr_mm": round(lr_mm, 1), "fg_ap_mm": round(ap_mm, 1),
            "fill_frac": round(fill, 4),
            "z0": z0, "z1": z1, "n_slices": int(im_u.shape[2]),
            "tumor_slices": int(len(zs)), "tumor_vox": t_after,
            "tumor_clipped_frac": round(1 - t_after / max(t_before, 1), 4),
            "int_p1": p1, "int_p50": p50, "int_p99": p99,
            "int_min": i_min, "int_max": i_max,
            "u16_scale": scale, "u16_offset": offset, "u16_rescaled": scaled,
        }))

    if not results:
        return [], {**base, "skip": "tumor_outside_fov"}
    return results, None


print("funksiyalar tayyor")

## 5. Sinov ishlov (dry run)

To'liq ishga tushirishdan oldin har manbadan 2 tadan holat. Diqqat:
`slice_axis_raw` ustuni manbalar orasida farq qilishi kerak (0 yoki 2) —
izotrop resampling shu farqni tekislaydi.

In [ ]:
by_src = collections.defaultdict(list)
for c in sorted(complete):
    by_src[source_of(c)].append(c)
probe = [c for s in sorted(by_src) for c in by_src[s][:2]]

rows, skips, cache = [], [], {}
for cid in probe:
    t = time.time()
    results, skip = process_case(cid, scratch)
    el = round(time.time() - t, 2)
    if skip:
        skips.append(skip)
        print(f"{cid:<14} O'TKAZILDI: {skip['skip']}")
        continue
    for data, meta in results:
        meta["sec"] = el / len(results)
        meta["mb"] = round((data[0].nbytes + data[1].nbytes) / 2**20, 2)
        cache[meta["key"]] = data
        rows.append(meta)
        print(f"{meta['key']:<16} {'ikki' if meta['bilateral'] else 'bitta':<5} ko'krak  "
              f"{meta['n_slices']:>4} kesim  to'ldirilganlik {meta['fill_frac']:.2f}  "
              f"{meta['mb']:>6} MB  {el:>5}s", flush=True)

import pandas as pd
probe_df = pd.DataFrame(rows)
cols = [c for c in ["key","source","side","bilateral","raw_shape","slice_axis_raw",
                    "fg_lr_mm","fg_ap_mm","fill_frac","n_slices","tumor_slices",
                    "tumor_clipped_frac","int_p99","u16_rescaled","mb"] if c in probe_df.columns]
display(probe_df[cols])

n_out = len(rows) / max(len(probe) - len(skips), 1)       # holatga o'rtacha namuna
est_gb  = probe_df["mb"].mean() * len(complete) * n_out / 1024
est_min = probe_df["sec"].sum() / max(len(probe), 1) * len(complete) / 60
print(f"\nHar holatga o'rtacha namuna: {n_out:.2f}")
print(f"Taxminiy to'liq hajm: {est_gb:.1f} GB  (/kaggle/working chegarasi 20 GB)")
print(f"Taxminiy vaqt       : {est_min:.0f} daqiqa")

if est_gb > 16:
    print("\n[!] Hajm chegaraga yaqin — OUT_SIZE=160 yoki TARGET_SPACING=2.0 qiling.")
if probe_df["tumor_clipped_frac"].max() > 0.02:
    print("\n[!] O'sma FOV chetida qirqilgan — OUT_SIZE ni oshiring.")

# to'ldirilganlik manbalar bo'yicha o'xshash bo'lsa, FOV soxta signali kamaygan
print("\nTo'ldirilganlik (kadrning to'qima bilan band ulushi) manba bo'yicha:")
print(probe_df.groupby("source")["fill_frac"].agg(["mean", "min", "max"]).round(3))

# bo'lish chegarasi xavfsizligini tekshirish
lo = probe_df.loc[probe_df["bilateral"], "fg_lr_mm"]
hi = probe_df.loc[~probe_df["bilateral"], "fg_lr_mm"]
print(f"\nL-R to'qima kengligi:  ikki ko'krakli {lo.min() if len(lo) else '-'}-{lo.max() if len(lo) else '-'} mm"
      f"   bitta ko'krakli {hi.min() if len(hi) else '-'}-{hi.max() if len(hi) else '-'} mm")
if len(lo) and len(hi):
    gap = lo.min() - hi.max()
    print(f"Chegara {BILATERAL_MM} mm,  ajralish oralig'i {gap:.0f} mm",
          "— xavfsiz" if gap > 40 else "— [!] TOR, chegarani qayta ko'ring")

### Vizual tekshiruv

Endi barcha manbalar **bir xil anatomik tekislikda** (aksial) va bir xil
fizik masshtabda bo'lishi kerak. Ko'krak anatomiyasi tanish ko'rinishda,
maska o'sma ustida bo'lsa — geometriya to'g'ri.

In [ ]:
import matplotlib.pyplot as plt

show = list(cache)[:6]
ncol = min(3, len(show)); nrow = (len(show) + ncol - 1) // ncol
fig, axes = plt.subplots(nrow, ncol, figsize=(4.2*ncol, 4.5*nrow), squeeze=False)

for ax, key in zip(axes.ravel(), show):
    im, mk = cache[key]
    z = int(np.argmax(mk.sum(axis=(0, 1))))
    sl = im[:, :, z].astype(np.float32)
    lo, hi = np.percentile(sl, [1, 99.5])
    ax.imshow(np.rot90(np.clip(sl, lo, hi)), cmap="gray")
    ax.contour(np.rot90(mk[:, :, z]), levels=[0.5], colors="red", linewidths=1.2)
    m = next(r for r in rows if r["key"] == key)
    ax.set_title(f"{m['source']} · {key}\nz={z}/{m['n_slices']}  "
                 f"to'ldirilganlik={m['fill_frac']:.2f}", fontsize=9)
    ax.axis("off")
for ax in axes.ravel()[len(show):]:
    ax.axis("off")

plt.tight_layout(); plt.show()

## 6. To'liq ishlov

Sinov to'g'ri bo'lsa: 1-hujayrada `DRY_RUN = None`, kernel Restart, Run All.

In [ ]:
assert DRY_RUN is None, "1-hujayrada DRY_RUN = None qilib, kernelni qayta ishga tushiring"

targets = sorted(complete)
handles, meta_rows, skipped = {}, [], []
t0 = time.time()

try:
    for i, cid in enumerate(targets, 1):
        results, skip = process_case(cid, scratch)
        if skip:
            skipped.append(skip)
        for data, meta in results:
            im, mk = data
            src = meta["source"]
            if src not in handles:
                handles[src] = h5py.File(os.path.join(OUTDIR, f"mamamia_{src}.h5"), "w")
            g = handles[src].create_group(meta["key"])
            g.create_dataset("img", data=im, compression="lzf", chunks=(OUT_SIZE, OUT_SIZE, 1))
            g.create_dataset("msk", data=mk, compression="lzf", chunks=(OUT_SIZE, OUT_SIZE, 1))
            meta_rows.append(meta)
        del results

        if i % 25 == 0 or i == len(targets):
            el = time.time() - t0
            used = sum(os.path.getsize(os.path.join(OUTDIR, f))
                       for f in os.listdir(OUTDIR)) / 2**30
            print(f"{i:>5}/{len(targets)}  {el/60:5.1f} daq  "
                  f"qolgan ~{el/i*(len(targets)-i)/60:5.1f} daq  "
                  f"namuna {len(meta_rows):>5}  disk {used:4.1f} GB  "
                  f"o'tkazilgan {len(skipped)}", flush=True)
            gc.collect()
            if used > 18:
                print("[!] Disk chegarasiga yaqin — to'xtatildi."); break
finally:
    for h in handles.values():
        h.close()

print(f"\nTugadi: {len(meta_rows)} namuna ({len(targets)} holatdan), "
      f"{len(skipped)} o'tkazildi, {(time.time()-t0)/60:.1f} daqiqa")

## 7. Metadata va yakuniy tekshiruv

In [ ]:
import pandas as pd

meta_df = pd.DataFrame(meta_rows)
meta_df.to_csv(os.path.join(OUTDIR, "metadata.csv"), index=False)

if skipped:
    pd.DataFrame(skipped).to_csv(os.path.join(OUTDIR, "skipped.csv"), index=False)
    print("O'tkazilganlar:", dict(collections.Counter(s["skip"] for s in skipped)))

with open(os.path.join(OUTDIR, "preprocess_config.json"), "w") as f:
    json.dump({"repo": REPO_ID, "target_spacing": TARGET_SPACING,
               "out_size": OUT_SIZE, "margin": MARGIN,
               "bilateral_mm": BILATERAL_MM,
               "fov_mm": OUT_SIZE * TARGET_SPACING,
               "n_samples": len(meta_df),
               "n_cases": int(meta_df["case"].nunique()),
               "date": time.strftime("%Y-%m-%d")}, f, indent=2)

print("\nManba bo'yicha:")
print(meta_df.groupby("source").agg(
    namuna=("key", "count"),
    holat=("case", "nunique"),
    ikki_kokrak=("bilateral", "sum"),
    tuldirilganlik=("fill_frac", "mean"),
    kesim=("n_slices", "mean"),
    p99=("int_p99", "mean"),
    qirqilgan=("tumor_clipped_frac", "max"),
).round(3))

print("\nXom kesim o'qi manba bo'yicha (izotropdan oldingi holat):")
print(meta_df.groupby("source")["slice_axis_raw"].value_counts().unstack(fill_value=0))

# --- bo'lish chegarasining o'zini tekshirish ---
pc = meta_df.drop_duplicates("case")
w = np.sort(pc["fg_lr_mm"].values)
d = np.diff(w)
i = int(np.argmax(np.where((w[:-1] > 200) & (w[:-1] < 320), d, -1)))
print(f"\nBo'lish chegarasi tekshiruvi:")
print(f"  ishlatilgan chegara      : {BILATERAL_MM} mm")
print(f"  taqsimotdagi eng katta bo'shliq (200-320 mm oralig'ida): "
      f"{w[i]:.1f} -> {w[i+1]:.1f}  ({d[i]:.1f} mm)")
if not (w[i] <= BILATERAL_MM <= w[i+1]):
    print(f"  [!] Chegara bo'shliq ichida EMAS — {(w[i]+w[i+1])/2:.0f} mm ga o'zgartiring "
          f"va qayta ishlating.")
else:
    print(f"  [ok] Chegara bo'shliq ichida — bo'lish qarori ishonchli.")

dup = meta_df[meta_df.duplicated("case", keep=False)]
print(f"\nIkkiga bo'lingan (ikki tomonlama o'sma): {dup['case'].nunique()} holat")
if len(dup):
    print(dup.groupby("source")["case"].nunique().to_string())

print("\nFayl hajmlari:")
tot = 0
for f in sorted(os.listdir(OUTDIR)):
    sz = os.path.getsize(os.path.join(OUTDIR, f)) / 2**20
    tot += sz
    print(f"  {f:<28} {sz:>9.1f} MB")
print(f"  {'JAMI':<28} {tot/1024:>9.2f} GB")

## 8. Keyingi qadam — Kaggle Dataset qilib saqlash

1. **Save Version** (Save & Run All) qiling.
2. Tugagach: Output → **New Dataset**, nomi `mamamia-preprocessed`,
   ko'rinishi **Private** (CC BY-NC litsenziyasi qayta tarqatishga ruxsat bermaydi).
3. Keyingi notebook'larda **Add Input** orqali ulaymiz — 20 GB qayta yuklanmaydi.

Shundan keyin `03_baseline_segmentation`: leave-one-source-out protokoli va
U-Net baseline.

### Maqola uchun eslatma

I-SPY1, I-SPY2 va NACT — o'zlari ko'p markazli ACRIN sinovlari, ya'ni bu
guruhlar bitta klinikaga teng emas. Shuning uchun leave-one-source-out ni
*domen siljishining proksisi* (boshqa sinov, protokol, davr) deb atash kerak;
haqiqiy tashqi validatsiya BreastDM orqali bo'ladi — boshqa davlat, boshqa
skaner. `slice_axis_raw` va `raw_spacing` ustunlari esa domen siljishining
miqdoriy dalili sifatida maqolaning Dataset bo'limiga kiradi.